# Prepare binary genre features

Replace `artist_genres` with 1,000 binary columns for the most frequent training genres and `genre_unknown`. Save a copy to `data/processed/train.csv` and the selected vocabulary to `models/genre_vocabulary.json`.

All other column values and the row order are preserved. Raw data remains unchanged. Run the cells from top to bottom; this notebook can run independently of the analysis notebook.

The CSV is about 735 MiB because it stores 1,000 genre indicators for every track. The export streams rows to avoid loading the entire expanded table into memory.

## 1. Set up paths and parameters

Change `top_n` here when comparing vocabulary sizes. The default reproduces the existing 1,000-genre preparation.

In [1]:
from pathlib import Path
import csv
from collections import Counter

# Locate the project from either the project root or the notebooks directory.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "data/raw/train.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from Projeto/CD or its notebooks directory.")
input_path = PROJECT_ROOT / "data/raw/train.csv"
print(f"Source: {input_path}")

import json

output_path = PROJECT_ROOT / "data/processed/train.csv"
top_n = 1000

input_path = input_path.resolve()
output_path = output_path.resolve()
raw_dir = (PROJECT_ROOT / "data/raw").resolve()
if output_path == input_path or output_path.is_relative_to(raw_dir):
    raise ValueError("Output must be separate from the source and outside data/raw/.")
if top_n < 1:
    raise ValueError("The number of genres must be positive.")


Source: /home/goncalotorcato/ist1109358/CD/Projeto/CD/data/raw/train.csv


## 2. Count genres in the training data

Split lists on `|`, strip surrounding spaces, and count a genre at most once per track.

In [2]:
def parse_genres(value):
    return {genre.strip() for genre in (value or "").split("|") if genre.strip()}

counts = Counter()
with input_path.open(encoding="utf-8-sig", newline="") as handle:
    reader = csv.reader(handle)
    original_columns = next(reader)
    genre_index = original_columns.index("artist_genres")
    for row in reader:
        counts.update(parse_genres(row[genre_index]))


print(f"Distinct genres: {len(counts):,}")


Distinct genres: 3,886


## 3. Select the vocabulary and build column names

Sort by decreasing frequency and break ties alphabetically. Column names keep the original genre text with a `genre_` prefix.

In [3]:
if len(counts) < top_n:
    raise ValueError(f"Requested {top_n} genres, but only {len(counts)} were found.")
# Alphabetical ordering makes frequency ties reproducible.
selected = sorted(counts, key=lambda genre: (-counts[genre], genre))[:top_n]
genre_columns = [f"genre_{genre}" for genre in selected]
new_columns = genre_columns + ["genre_unknown"]
other_columns = original_columns[:genre_index] + original_columns[genre_index + 1:]
if len(set(new_columns + other_columns)) != len(new_columns + other_columns):
    raise ValueError("Generated genre column names collide with existing columns.")
header = original_columns[:genre_index] + new_columns + original_columns[genre_index + 1:]
positions = {genre: index for index, genre in enumerate(selected)}


print(f"Selected genres: {len(selected):,}; output columns: {len(header):,}")
print(genre_columns[:10])


Selected genres: 1,000; output columns: 1,024
['genre_karaoke', 'genre_rap', 'genre_pop', 'genre_rock', 'genre_hip hop', 'genre_classical', 'genre_pop rap', 'genre_dance pop', 'genre_trap', 'genre_classic rock']


## 4. Write the processed training copy

For each track, set a selected genre to 1 when it is present and to 0 otherwise. Set `genre_unknown` to 1 only when no genres were provided.

A track with known genres outside the top 1,000 has zero selected genre indicators and `genre_unknown = 0`. Running this cell regenerates the processed CSV.

In [4]:
output_path.parent.mkdir(parents=True, exist_ok=True)
template = [0] * len(new_columns)
rows_written = 0
unknown_rows = 0
with input_path.open(encoding="utf-8-sig", newline="") as source:
    reader = csv.reader(source)
    next(reader)
    with output_path.open("w", encoding="utf-8", newline="") as destination:
        writer = csv.writer(destination)
        writer.writerow(header)
        for row in reader:
            genres = parse_genres(row[genre_index])
            indicators = template.copy()
            for genre in genres:
                if genre in positions:
                    indicators[positions[genre]] = 1
            # Known genres outside the top N do not count as missing information.
            indicators[-1] = int(not genres)
            unknown_rows += indicators[-1]
            writer.writerow(row[:genre_index] + indicators + row[genre_index + 1:])
            rows_written += 1


## 5. Save the vocabulary and report the result

Save the selected genres and their column names so later datasets can use exactly the same columns.

In [ ]:
vocabulary = {
    "source": input_path.name,
    "selection": "Track frequency descending, then genre name ascending for ties",
    "top_n": top_n,
    "rows": rows_written,
    "unknown_rows": unknown_rows,
    "unknown_column": "genre_unknown",
    "genres": [
        {"genre": genre, "column": column, "track_count": counts[genre]}
        for genre, column in zip(selected, genre_columns)
    ],
}
vocabulary_path = PROJECT_ROOT / "models/genre_vocabulary.json"
vocabulary_path.parent.mkdir(parents=True, exist_ok=True)
vocabulary_path.write_text(json.dumps(vocabulary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Saved {rows_written:,} rows and {len(header):,} columns to {output_path}")
print(f"Genre columns: {top_n:,}; missing-genre rows: {unknown_rows:,}")
print(f"Vocabulary: {vocabulary_path}")


## Model evaluation

This notebook reproduces the requested copy using the full training CSV. When comparing models, fit vocabulary selection using only the training split, then apply that vocabulary to the validation split. Do not select a separate vocabulary for validation or test data.